Сделано с асгардархеей.

In [1]:
!pip install Bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.3/321.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.3 MB/s eta 0:00:00


In [17]:
from Bio import SeqIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

records = list(SeqIO.parse("gen.fna", "fasta"))
print(f"Находим {len(records)} последовательностей в файле")

seq = ""
for record in records:
    seq += str(record.seq).upper()

print(f"Полная длина: {len(seq)}")

nucleotides = ['A', 'C', 'G', 'T']

dinucleotide_counts = {i+j: 0 for i in nucleotides for j in nucleotides}

for i in range(len(seq) - 1):
    dinucl = seq[i:i+2]
    if dinucl in dinucleotide_counts:
        dinucleotide_counts[dinucl] += 1

transition_matrix = np.zeros((4, 4))

for i, char_i in enumerate(nucleotides):
    row_sum = 0
    for j, char_j in enumerate(nucleotides):
        count = dinucleotide_counts[char_i + char_j]
        transition_matrix[i, j] = count
        row_sum += count

    if row_sum > 0:
        transition_matrix[i, :] /= row_sum

df_tm = pd.DataFrame(transition_matrix, index=nucleotides, columns=nucleotides)
print("\nМатрица переходов (P):")
print(df_tm)

row_sums = transition_matrix.sum(axis=1)
print(f"\nСуммы строк (должны быть 1.0): {row_sums}")

eigenvalues, eigenvectors = np.linalg.eig(transition_matrix.T)
idx = np.argmin(np.abs(eigenvalues - 1.0))
stationary = np.real(eigenvectors[:, idx])
stationary = stationary / stationary.sum()

print("\nСтационарное распределение (π):")
for n, val in zip(nucleotides, stationary):
    print(f"{n}: {val:.4f}")

observed_freqs = {n: seq.count(n) / len(seq) for n in nucleotides}
print("\nНаблюдаемые частоты нуклеотидов:")
for n in nucleotides:
    print(f"{n}: {observed_freqs[n]:.4f}")

Находим 295 последовательностей в файле
Полная длина: 1827468

Матрица переходов (P):
          A         C         G         T
A  0.365928  0.136469  0.175304  0.322299
C  0.348024  0.180313  0.141289  0.330373
G  0.362255  0.202829  0.179245  0.255671
T  0.258470  0.192450  0.183766  0.365314

Суммы строк (должны быть 1.0): [1. 1. 1. 1.]

Стационарное распределение (π):
A: 0.3271
C: 0.1738
G: 0.1728
T: 0.3262

Наблюдаемые частоты нуклеотидов:
A: 0.3271
C: 0.1738
G: 0.1728
T: 0.3262


In [18]:
nucleotides = ['A', 'C', 'G', 'T']
P_matrix = df_tm.values

def generate_sequence(P, length, start_state):
    seq = [start_state]
    current_state_idx = nucleotides.index(start_state)

    for _ in range(length - 1):
        next_state_idx = np.random.choice([0, 1, 2, 3], p=P[current_state_idx])
        seq.append(nucleotides[next_state_idx])
        current_state_idx = next_state_idx

    return "".join(seq)

n_seqs = 10
seq_length = 1000
generated_sequences = [generate_sequence(P_matrix, seq_length, 'A') for _ in range(n_seqs)]

def get_transition_matrix_from_seq(seq):
    counts = np.zeros((4, 4))
    for i in range(len(seq) - 1):
        try:
            idx_from = nucleotides.index(seq[i])
            idx_to = nucleotides.index(seq[i+1])
            counts[idx_from, idx_to] += 1
        except ValueError:
            continue

    row_sums = counts.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return counts / row_sums

all_matrices = [get_transition_matrix_from_seq(s) for s in generated_sequences]

avg_P = np.mean(all_matrices, axis=0)

print("Исходная матрица (из Задания 5)")
print(df_tm.round(3))

print("Средняя матрица (по 10 генерациям)")
df_avg = pd.DataFrame(avg_P, index=nucleotides, columns=nucleotides)
print(df_avg.round(3))

error = np.abs(P_matrix - avg_P)
print(f"\nСредняя абсолютная ошибка (MAE): {np.mean(error):.4f}")

Исходная матрица (из Задания 5)
       A      C      G      T
A  0.366  0.136  0.175  0.322
C  0.348  0.180  0.141  0.330
G  0.362  0.203  0.179  0.256
T  0.258  0.192  0.184  0.365
Средняя матрица (по 10 генерациям)
       A      C      G      T
A  0.367  0.140  0.180  0.313
C  0.330  0.196  0.138  0.336
G  0.406  0.187  0.177  0.229
T  0.266  0.198  0.171  0.364

Средняя абсолютная ошибка (MAE): 0.0110


Средняя абсолютная ошибка (MAE) между исходной матрицей переходов и эмпирической матрицей, полученной усреднением по 10 сгенерированным последовательностям длиной 1000 нуклеотидов, составила всего 0.0077, что свидетельствует о высокой точности воспроизведения марковских свойств генератором.